# Omnilingual ASR — Smoke Test (Phase 1, ADR-0003)

**But** : vérifier qu'Omnilingual ASR 300M charge et transcrit 1 audio dioula sans crash, reproductiblement, sur Colab T4.

**Ne PAS faire dans ce notebook** : benchmark vs NeMo (Phase 3), création du provider Python (Phase 2), intégration en chain (Phase 4).

**Critères de sortie ADR-0003 Phase 1** (validés 2026-05-04 sur audio `common_voice_dyu_38389110.mp3`) :
- ✅ Notebook exécute complètement sans crash sur Colab T4
- ✅ Omnilingual 300M charge en < 60 s (mesuré 27.8 s)
- ✅ 1 audio dioula transcrit (output non vide)
- ⏳ Doc install reproduit par Ruben sur 2ème session Colab (à valider)

**Procédure stricte** : exécuter cellules dans l'ordre. **Restart de session OBLIGATOIRE** entre Cellule 1 (install) et Cellule 2 (test) pour résoudre l'ABI numpy.

**Référence** : [ADR-0002](../../docs/adr/0002-ajout-provider-omnilingual.md), [ADR-0003](../../docs/adr/0003-plan-ajout-omnilingual.md), [doc env-setup](../../docs/benchmarks/0002-omnilingual-env-setup.md)

## Cellule 0 — Vérification du runtime Colab

Avant tout, confirmer qu'on est bien sur GPU T4 (Runtime → Change runtime type → T4 GPU).

In [ ]:
import sys
import platform
import subprocess

print(f"Python   : {sys.version}")
print(f"Platform : {platform.platform()}")

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
        text=True,
    )
    print("\n--- GPU détecté ---")
    print(out)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("\nERREUR : pas de GPU NVIDIA détecté.")
    print("Activer GPU T4 via : Runtime → Change runtime type → T4 GPU")
    raise SystemExit(1)

## Cellule 1 — Installation propre `omnilingual-asr` + stack PyTorch aligné

**Versions validées 2026-05-04 sur Colab T4** :

| Package | Version | Note |
|---|---|---|
| `omnilingual-asr` | 0.1.0 | Apache 2.0 |
| `fairseq2` | 0.6 | downgrade torch automatique |
| `torch` | 2.8.0 | Compat fairseq2 (Colab démarre torch 2.10/2.11) |
| `torchaudio` | 2.8.0 | Aligné avec torch 2.8.0 + CUDA 12 |
| `torchvision` | 0.23.0 | Aligné avec torch 2.8.0 + CUDA 12 |

**Pourquoi le stack est réinstallé** : Colab préinstalle `torchaudio 2.10/2.11` et `torchvision 0.25` compilés pour CUDA 13. Mais fairseq2 0.6 downgrade `torch` à 2.8.0 (CUDA 12). Cascade de bugs (`libcudart.so.13`, `torchvision::nms`) si on ne réaligne pas tout.

**Durée** : 5-7 minutes (downloads ~1.5 GB).

In [ ]:
# Install propre Omnilingual ASR (procédure validée Phase 1)
!pip install --quiet omnilingual-asr fairseq2
!pip uninstall -y torch torchaudio torchvision
!pip install --quiet torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0

# Vérification versions installées
import importlib.metadata as md
print("\n=== Versions installées ===")
for pkg in ["omnilingual-asr", "fairseq2", "torch", "torchaudio", "torchvision"]:
    try:
        print(f"  {pkg:20s} : {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:20s} : NON INSTALLÉ")

## ⚠️ RESTART OBLIGATOIRE ICI

**Avant d'exécuter la Cellule 2, redémarrer la session** :

1. Menu **Exécution** → **Redémarrer la session** (⚠️ PAS "Réinitialiser tout l'environnement")
2. Confirmer **Oui**
3. Attendre 10-15 sec que ✅ vert revienne en haut à droite

**Raison** : numpy 2.x est chargé en mémoire au démarrage Colab par d'autres modules. fairseq2 a downgradé numpy à 1.26.4 sur disque. Sans restart, les modules torch._dynamo plantent avec `numpy.dtype size changed`.

**Différence critique** :
- ✅ "Redémarrer la session" = vide la mémoire Python, garde les packages
- ❌ "Réinitialiser tout l'environnement" = SUPPRIME tous les packages → 7 min reperdues

## Cellule 2 — Vérification environnement PyTorch + CUDA

Confirmer que PyTorch voit le GPU après restart.

In [ ]:
import torch

print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"CUDA version   : {torch.version.cuda}")
print(f"GPU count      : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 0 name     : {torch.cuda.get_device_name(0)}")
    print(f"GPU 0 memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

assert torch.cuda.is_available(), "GPU CUDA requis pour Omnilingual — activer T4 GPU dans Colab"

## Cellule 3 — Import Omnilingual + vérification `dyu_Latn` présent

Confirmer que la langue cible (dioula CI = `dyu_Latn`) figure dans la liste des 1672 langues supportées.

**Chemin import validé** : `omnilingual_asr.models.wav2vec2_llama.lang_ids.supported_langs`

In [ ]:
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs

print(f"✓ Import OK")
print(f"  Total langues : {len(supported_langs)}")
print(f"  Type          : {type(supported_langs).__name__}")
print()

# Vérifier les langues pertinentes pour Wourri (cf. ADR-0002)
for code in ["dyu_Latn", "bam_Latn", "bci_Latn", "ann_Latn", "fra_Latn"]:
    present = code in supported_langs
    marker = "✓" if present else "✗"
    print(f"  {marker} {code}")

## Cellule 4 — Chargement du modèle Omnilingual CTC 300M

**API validée** : `ASRInferencePipeline(model_card="omniASR_CTC_300M")` (note : pas `_v2`).

Mesurer le temps de chargement (cible ADR-0003 : < 60 s).

**Première exécution** : ~1-3 min (download HF Hub ~1.2 GB). **Exécutions suivantes** : < 30 s (cache local Colab).

In [ ]:
import time
import psutil
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

process = psutil.Process()
ram_before = process.memory_info().rss / 1e9
torch.cuda.reset_peak_memory_stats()
vram_before = torch.cuda.memory_allocated() / 1e9

print("Téléchargement et chargement du modèle Omnilingual CTC 300M...")
t0 = time.time()
pipeline = ASRInferencePipeline(model_card="omniASR_CTC_300M")
t_load = time.time() - t0

ram_after = process.memory_info().rss / 1e9
vram_after = torch.cuda.memory_allocated() / 1e9
vram_peak = torch.cuda.max_memory_allocated() / 1e9

print(f"\n=== Métriques chargement ===")
print(f"Temps : {t_load:.1f} s (cible ADR-0003 : < 60 s)")
print(f"RAM   : {ram_before:.2f} → {ram_after:.2f} GB (delta {ram_after - ram_before:+.2f})")
print(f"VRAM  : {vram_before:.2f} → {vram_after:.2f} GB (peak {vram_peak:.2f})")

## Cellule 5 — Préparer un audio dioula

**Approche choisie** : upload manuel d'un MP3 Common Voice dyu v24 (drag-drop dans `/content/` via le panneau Files de Colab).

**Pourquoi pas `datasets.load_dataset("mozilla-foundation/common_voice_24_0", "dyu")`** : ce dataset est *gated* (nécessite token HF + acceptation licence).

**Procédure** :
1. Sur ton PC local, prendre un MP3 dans `cv-corpus-24.0-2025-12-05-dyu/clips/`
2. Noter sa transcription depuis `validated.tsv`
3. Drag-drop le MP3 dans Colab (panneau Files → /content/)
4. Mettre à jour `AUDIO_PATH` et `REFERENCE` ci-dessous

In [ ]:
import torchaudio

# ⚠️ Adapter selon le fichier que tu as uploadé
AUDIO_PATH = "/content/common_voice_dyu_38389110.mp3"
REFERENCE = "I ka kɛnɛ wa?"

audio_tensor, sample_rate = torchaudio.load(AUDIO_PATH)
if audio_tensor.shape[0] > 1:
    audio_tensor = audio_tensor.mean(dim=0, keepdim=True)

duration_audio = audio_tensor.shape[-1] / sample_rate

print(f"=== Audio chargé ===")
print(f"Fichier      : {AUDIO_PATH}")
print(f"Sample rate  : {sample_rate} Hz")
print(f"Durée        : {duration_audio:.2f} s")
print(f"Référence    : {REFERENCE}")
print(f"\n→ Contrainte Omnilingual CTC < 40s : {'✓ OK' if duration_audio < 40 else '✗ Trop long'}")

## Cellule 6 — Transcription dioula

**API validée** : `pipeline.transcribe([chemin_fichier], lang=["dyu_Latn"], batch_size=1)`.

Le pipeline gère le resampling et le format en interne — pas besoin de pré-traiter l'audio.

In [ ]:
print(f"Transcription en cours (lang='dyu_Latn')...")

t0 = time.time()
transcriptions = pipeline.transcribe(
    [AUDIO_PATH],
    lang=["dyu_Latn"],
    batch_size=1,
)
t_inf = time.time() - t0
rtf = t_inf / duration_audio

print(f"\n=== Résultat ===")
print(f"Référence     : {REFERENCE}")
print(f"Transcription : {transcriptions[0]}")
print(f"\n=== Métriques ===")
print(f"Latence       : {t_inf:.2f} s pour {duration_audio:.2f} s d'audio")
print(f"RTF           : {rtf:.3f} (cible benchmark < 0.5)")
print(f"\n→ Match exact : {'✓ OUI' if transcriptions[0].strip() == REFERENCE else '✗ NON (qualité à mesurer en Phase 3 sur 1B + 20 audios)'}")

## Cellule 7 — Validation des critères de sortie Phase 1

Récapitulatif des 4 critères ADR-0003.

In [ ]:
criteres = {
    "Notebook exécute sans crash"          : t_load > 0 and t_inf > 0,
    "Modèle 300M charge en < 60 s"         : t_load < 60,
    "1 audio dioula transcrit (non vide)"  : len(transcriptions[0].strip()) > 0,
    "Reproduit sur 2ème session Colab"     : None,  # à valider manuellement
}

print("=== Critères de sortie ADR-0003 Phase 1 ===")
for critere, statut in criteres.items():
    if statut is True:
        marker = "✅"
    elif statut is False:
        marker = "❌"
    else:
        marker = "⏳"
    print(f"  {marker}  {critere}")

n_ok = sum(1 for v in criteres.values() if v is True)
print(f"\n→ {n_ok}/4 critères validés automatiquement")
print("→ Reporter ces résultats dans docs/benchmarks/0002-omnilingual-env-setup.md")